In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path for modular imports from src/
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
from src.data_loader import load_combined_dataset
from src.preprocessing import prepare_dataframe, create_stratified_splits, NewsDataset
from src.model import get_device, get_model_and_tokenizer, save_model_and_tokenizer
from src.train import set_seed, train_model
from src.evaluate import evaluate_model

device, device_name = get_device()
print(f"PyTorch version: {torch.__version__}")
print(f"Active device: {device_name}")

In [ ]:
# Load and inspect combined dataset
raw_df = load_combined_dataset(shuffle=True, random_state=42)
print(f"Raw dataset records: {len(raw_df)}")
print(raw_df.head(2))

In [ ]:
# Preprocess dataset: strip datelines and drop duplicates to prevent cross-split leakage
prepared_df = prepare_dataframe(raw_df, strip_dateline=True, drop_duplicates=True)
print(f"Cleaned dataset records: {len(prepared_df)}")
print(f"Sample formatted input:\n{prepared_df.loc[0, 'formatted_text'][:200]}...")

In [ ]:
# Create stratified 80 / 10 / 10 splits
train_df, val_df, test_df = create_stratified_splits(
    prepared_df, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, random_state=42
)
print(f"Train set size: {len(train_df)} (Fake={train_df['label'].sum()})")
print(f"Val set size:   {len(val_df)} (Fake={val_df['label'].sum()})")
print(f"Test set size:  {len(test_df)} (Fake={test_df['label'].sum()})")

In [ ]:
# Verify tokenizer behavior on sample input
_, tokenizer = get_model_and_tokenizer("bert-base-uncased")
sample_encoding = tokenizer(
    prepared_df.loc[0, "formatted_text"],
    truncation=True,
    max_length=256,
    padding="max_length",
    return_tensors="pt"
)
print("Input IDs shape:", sample_encoding["input_ids"].shape)
print("Attention Mask shape:", sample_encoding["attention_mask"].shape)
print("Decoded token start:", tokenizer.convert_ids_to_tokens(sample_encoding["input_ids"][0][:15]))

In [ ]:
# Train BERT model using src.train pipeline
# Configured for 6 GB GPU with mixed precision
training_results = train_model(
    model_name="bert-base-uncased",
    max_length=256,
    batch_size=16,
    learning_rate=2e-5,
    epochs=3,
    output_dir="../artifacts/model",
    metrics_dir="../artifacts/metrics",
    strip_dateline=True,
)
print("Training Summary:")
print(training_results)

In [ ]:
# Evaluate model on held-out test set
eval_results = evaluate_model(
    model_dir="../artifacts/model",
    test_data_path="../artifacts/metrics/test_set.csv",
    metrics_dir="../artifacts/metrics",
)
print("\nHeld-out Test Evaluation Metrics:")
for metric, val in eval_results.items():
    if metric not in ["confusion_matrix", "classification_report"]:
        print(f"{metric}: {val}")